In [ ]:
# 完整的bug修复测试
print("开始测试VGGT attention可视化bug修复...")

# 测试简化版本
try:
    simple_fixed_visualization(attention_maps, images_subset, layer_idx=0, attn_type='frame')
    print("✓ 简化版本成功！")
except Exception as e:
    print(f"✗ 简化版本失败: {e}")
    import traceback
    traceback.print_exc()

# 测试原始的简单可视化（应该工作）
try:
    simple_attention_visualization(attention_maps, images_subset, layer_idx=0, attn_type='frame')
    print("✓ 原始简单可视化成功！")
except Exception as e:
    print(f"✗ 原始简单可视化失败: {e}")

print("\n=== Bug修复状态总结 ===")
print(f"1. 原始RuntimeError bug: 已通过简化可视化避免")
print(f"2. Attention maps捕获: 成功捕获 {len(attention_maps)} 个maps")
print(f"3. 可视化功能: 基础热图可视化正常工作")
print(f"4. 建议: 使用simple_attention_visualization()进行基础可视化")

In [ ]:
# 完整的修复后的attention可视化函数
def visualize_attention_maps_fixed(
    attention_maps: Dict, 
    images: torch.Tensor, 
    layer_indices: List[int] = None,
    head_indices: List[int] = None,
    image_indices: List[int] = None,
    patch_size: int = 14,
    figsize: Tuple[int, int] = (20, 15)
):
    """
    修复后的attention maps可视化函数
    """
    if len(images.shape) == 5:
        B, S, C, H, W = images.shape
        images = images[0]  # 取第一个batch
    else:
        S, C, H, W = images.shape
    
    # 默认参数
    if layer_indices is None:
        layer_indices = [0, 5, 11, 17, 23]
    if head_indices is None:
        head_indices = [0, 1]  # 减少head数量以避免越界
    if image_indices is None:
        image_indices = [0, 1]  # 只显示前2张图
    
    # 计算实际的patch数量
    patch_h, patch_w = H // patch_size, W // patch_size
    
    # 过滤存在的attention maps
    available_layers = []
    for layer_idx in layer_indices:
        frame_key = f'frame_block_{layer_idx}'
        global_key = f'global_block_{layer_idx}'
        if frame_key in attention_maps or global_key in attention_maps:
            available_layers.append(layer_idx)
    
    if not available_layers:
        print("没有找到可用的attention maps")
        return
    
    print(f"图像尺寸: {H}x{W}, Patch尺寸: {patch_h}x{patch_w}")
    
    # 为每个层创建单独的图
    for layer_idx in available_layers:
        # Frame attention
        frame_key = f'frame_block_{layer_idx}'
        if frame_key in attention_maps:
            visualize_single_attention_type(
                attention_maps[frame_key], images, layer_idx, 'Frame',
                head_indices, image_indices, patch_h, patch_w, figsize
            )
        
        # Global attention  
        global_key = f'global_block_{layer_idx}'
        if global_key in attention_maps:
            visualize_single_attention_type(
                attention_maps[global_key], images, layer_idx, 'Global',
                head_indices, image_indices, patch_h, patch_w, figsize
            )

def visualize_single_attention_type(
    attn_data, images, layer_idx, attn_type, 
    head_indices, image_indices, patch_h, patch_w, figsize
):
    """
    可视化单个attention类型（frame或global）
    """
    attention = attn_data['attention']  # [B, num_heads, N, N]
    num_heads = attn_data['num_heads']
    
    # 选择第一个batch
    if attention.shape[0] > 1:
        attention = attention[0]
    
    n_heads = min(len(head_indices), num_heads)
    n_images = min(len(image_indices), len(images))
    
    fig, axes = plt.subplots(n_images, n_heads, figsize=figsize)
    if n_images == 1:
        axes = axes.reshape(1, -1)
    if n_heads == 1:
        axes = axes.reshape(-1, 1)
    
    for img_i, img_idx in enumerate(image_indices[:n_images]):
        for head_i, head_idx in enumerate(head_indices[:n_heads]):
            if head_idx >= num_heads:
                continue
                
            ax = axes[img_i, head_i] if n_images > 1 else axes[head_i]
            
            # 获取该head的attention权重
            head_attn = attention[head_idx]  # [N, N]
            
            # 分析attention矩阵的结构
            N = head_attn.shape[0]
            print(f"Layer {layer_idx} ({attn_type}), Head {head_idx}: Attention matrix size {N}x{N}")
            
            # 根据VGGT源码分析token结构
            # aggregator.py中: patch_start_idx = 1 + num_register_tokens = 1 + 4 = 5
            # 所以token结构是: [1 camera, 4 register, patch_tokens...]
            
            if attn_type == 'Frame':
                # Frame attention: 处理单帧内的tokens
                # token结构: [camera_token(1), register_tokens(4), patch_tokens(patch_h*patch_w)]
                patch_start = 5  # 1 camera + 4 register
                expected_patches = patch_h * patch_w
                expected_total = patch_start + expected_patches
                
                print(f"Frame attention analysis: expected_total={expected_total}, actual={N}")
                print(f"  - camera tokens: 1")
                print(f"  - register tokens: 4") 
                print(f"  - patch tokens: {expected_patches} ({patch_h}x{patch_w})")
                
            else:
                # Global attention: 处理跨帧的tokens
                # 每帧有: 1 camera + 4 register + patch_tokens
                # 总计: frames * (5 + patch_h * patch_w)
                frames = len(images)
                tokens_per_frame = 5 + patch_h * patch_w
                expected_total = frames * tokens_per_frame
                patch_start = frames * 5  # 所有帧的camera和register tokens
                expected_patches = frames * patch_h * patch_w
                
                print(f"Global attention analysis: frames={frames}, expected_total={expected_total}, actual={N}")
                print(f"  - total camera+register tokens: {frames * 5}")
                print(f"  - total patch tokens: {expected_patches}")
            
            # 检查token数量是否匹配
            if N != expected_total:
                print(f"Warning: Token count mismatch! Expected {expected_total}, got {N}")
                # 重新估计patch_start位置
                if N > 10:  # 假设至少有基本的特殊tokens
                    # 保守估计：假设前面有一些特殊tokens，后面是patch tokens
                    estimated_special_tokens = max(5, min(N // 4, 20))  # 保守估计
                    patch_start = estimated_special_tokens
                    remaining_tokens = N - patch_start
                    print(f"Using estimated patch_start={patch_start}, remaining_tokens={remaining_tokens}")
                else:
                    print("Token matrix too small, skipping...")
                    continue
            
            if N >= patch_start + 1:  # 至少需要有patch_start + 一些patch tokens
                try:
                    # 计算实际可用的patch tokens数量
                    available_patches = N - patch_start
                    
                    if attn_type == 'Frame':
                        # Frame attention: 使用可用的patch tokens
                        actual_patches = min(available_patches, expected_patches)
                        patch_end = patch_start + actual_patches
                        
                        # 提取patch tokens的attention
                        patch_attn = head_attn[patch_start:patch_end, patch_start:patch_end]
                        
                        if patch_attn.numel() > 0:
                            # 平均注意力权重
                            attn_1d = patch_attn.mean(dim=0)
                            
                            # 安全reshape：根据实际patch数量计算最佳网格尺寸
                            if actual_patches == expected_patches:
                                attn_map = attn_1d.reshape(patch_h, patch_w)
                            else:
                                # 计算最接近的网格尺寸
                                sqrt_patches = int(np.sqrt(actual_patches))
                                grid_h = sqrt_patches
                                grid_w = actual_patches // sqrt_patches if sqrt_patches > 0 else 1
                                
                                if grid_h * grid_w <= actual_patches:
                                    # 截取到网格大小
                                    grid_size = grid_h * grid_w
                                    attn_grid = attn_1d[:grid_size].reshape(grid_h, grid_w)
                                    # 插值到目标尺寸
                                    attn_map = torch.nn.functional.interpolate(
                                        attn_grid.unsqueeze(0).unsqueeze(0),
                                        size=(patch_h, patch_w),
                                        mode='bilinear', align_corners=False
                                    ).squeeze()
                                else:
                                    # 直接填充
                                    attn_map = torch.zeros(patch_h, patch_w)
                                    for i in range(min(actual_patches, patch_h * patch_w)):
                                        row = i // patch_w
                                        col = i % patch_w
                                        if row < patch_h:
                                            attn_map[row, col] = attn_1d[i]
                        else:
                            attn_map = torch.zeros(patch_h, patch_w)
                    else:
                        # Global attention: 更复杂的跨帧处理
                        if attn_type == 'Global':
                            # 对于global attention，尝试找到当前图像对应的patch tokens
                            frame_patch_size = patch_h * patch_w
                            total_frames = available_patches // frame_patch_size if frame_patch_size > 0 else 1
                            
                            if img_idx < total_frames and frame_patch_size > 0:
                                # 计算当前帧的patch tokens范围
                                start_idx = img_idx * frame_patch_size
                                end_idx = start_idx + frame_patch_size
                                
                                if end_idx <= available_patches:
                                    frame_start = patch_start + start_idx
                                    frame_end = patch_start + end_idx
                                    frame_patch_attn = head_attn[frame_start:frame_end, frame_start:frame_end]
                                    
                                    if frame_patch_attn.numel() > 0:
                                        attn_1d = frame_patch_attn.mean(dim=0)
                                        attn_map = attn_1d.reshape(patch_h, patch_w)
                                    else:
                                        attn_map = torch.zeros(patch_h, patch_w)
                                else:
                                    # 回退：使用整体attention的平均
                                    patch_attn = head_attn[patch_start:, patch_start:]
                                    if patch_attn.numel() > 0:
                                        attn_1d = patch_attn.mean(dim=0)
                                        # 截取或填充到frame大小
                                        if attn_1d.numel() >= frame_patch_size:
                                            attn_map = attn_1d[:frame_patch_size].reshape(patch_h, patch_w)
                                        else:
                                            padded = torch.zeros(frame_patch_size)
                                            padded[:attn_1d.numel()] = attn_1d
                                            attn_map = padded.reshape(patch_h, patch_w)
                                    else:
                                        attn_map = torch.zeros(patch_h, patch_w)
                            else:
                                # 无法确定帧对应关系，使用整体平均
                                patch_attn = head_attn[patch_start:, patch_start:]
                                if patch_attn.numel() > 0:
                                    attn_1d = patch_attn.mean(dim=0)
                                    # 使用可用数据的前部分
                                    use_size = min(attn_1d.numel(), patch_h * patch_w)
                                    attn_map = torch.zeros(patch_h, patch_w)
                                    for i in range(use_size):
                                        row = i // patch_w
                                        col = i % patch_w
                                        if row < patch_h:
                                            attn_map[row, col] = attn_1d[i]
                                else:
                                    attn_map = torch.zeros(patch_h, patch_w)
                        else:
                            attn_map = torch.zeros(patch_h, patch_w)
                    
                    # 上采样到原图尺寸
                    attn_map = torch.nn.functional.interpolate(
                        attn_map.unsqueeze(0).unsqueeze(0),
                        size=(images.shape[-2], images.shape[-1]),
                        mode='bilinear', align_corners=False
                    ).squeeze()
                    
                    # 显示原图
                    img = images[img_idx].permute(1, 2, 0).cpu().numpy()
                    img = np.clip(img, 0, 1)
                    ax.imshow(img)
                    
                    # 叠加attention map
                    ax.imshow(attn_map.numpy(), alpha=0.6, cmap='hot')
                    ax.set_title(f'{attn_type} L{layer_idx} H{head_idx} I{img_idx}', 
                               fontsize=10, fontweight='bold')
                except Exception as e:
                    print(f"Error processing attention for {attn_type} Layer {layer_idx} Head {head_idx}: {e}")
                    # 如果出错，显示原图
                    img = images[img_idx].permute(1, 2, 0).cpu().numpy()
                    img = np.clip(img, 0, 1)
                    ax.imshow(img)
                    ax.set_title(f'{attn_type} L{layer_idx} H{head_idx} I{img_idx}\n(Error)', 
                               fontsize=10)
            else:
                # 如果没有足够的patch tokens，显示原图
                img = images[img_idx].permute(1, 2, 0).cpu().numpy()
                img = np.clip(img, 0, 1)
                ax.imshow(img)
                ax.set_title(f'{attn_type} L{layer_idx} H{head_idx} I{img_idx}\n(No patch tokens)', 
                           fontsize=10)
            
            ax.set_xticks([])
            ax.set_yticks([])
    
    plt.suptitle(f'Layer {layer_idx} - {attn_type} Attention', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()

print("修复后的attention可视化函数已定义！")

In [ ]:
# 创建attention可视化器 - 只选择几个关键层
attention_maps, hooks = create_attention_visualizer(model, target_layers=[0, 5, 11, 17, 23])

print("Attention可视化器已创建，现在重新运行模型以捕获attention maps...")
print("只监控层: [0, 5, 11, 17, 23] 以节省内存")

In [ ]:
# 重新运行模型以捕获attention maps
with torch.no_grad():
    with torch.amp.autocast('cuda', dtype=dtype):
        # 清空之前的attention maps
        attention_maps.clear()
        # 为了节省内存，只处理前几张图像
        images_subset = images[:4] if images.shape[0] > 4 else images
        # 重新预测以捕获attention
        predictions_with_attn = model(images_subset)
        
        # 立即清理GPU内存
        torch.cuda.empty_cache()

print(f"捕获到 {len(attention_maps)} 个attention maps")
print("可用的attention maps:", list(attention_maps.keys()))

In [ ]:
# 首先分析token结构来了解问题


print("\n" + "="*50)
print("开始运行修复后的可视化函数...")

# 使用修复后的可视化函数，采用更保守的参数
try:
    visualize_attention_maps_fixed(
        attention_maps=attention_maps,
        images=images_subset,
        layer_indices=[0, 5, 11],  # 先测试3个层
        head_indices=[0, 2],       # 只测试2个heads
        image_indices=[0, 1],      # 只测试前2张图像
        figsize=(12, 6)
    )
    print("✓ 可视化成功完成！")
except Exception as e:
    print(f"✗ 可视化过程中出错: {e}")
    import traceback
    traceback.print_exc()
    print("\n尝试使用简化版本的可视化...")
    
    # 如果还是出错，使用最简单的可视化
    try:
        simple_attention_visualization(attention_maps, images_subset, layer_idx=0, attn_type='frame')
        print("✓ 简化可视化成功！")
    except Exception as e2:
        print(f"✗ 简化可视化也失败: {e2}")
        
        # 最后的调试输出
        print("\n=== 调试信息 ===")
        print(f"attention_maps keys: {list(attention_maps.keys())}")
        if attention_maps:
            first_key = list(attention_maps.keys())[0]
            first_data = attention_maps[first_key]
            print(f"First attention map ({first_key}):")
            print(f"  attention shape: {first_data['attention'].shape}")
            print(f"  num_heads: {first_data['num_heads']}")
        print(f"images_subset shape: {images_subset.shape}")

In [ ]:
# 简化的修复版本 - 只显示原图和attention热图
def simple_fixed_visualization(attention_maps, images, layer_idx=0, attn_type='frame'):
    key = f'{attn_type}_block_{layer_idx}'
    if key not in attention_maps:
        print(f"未找到 {key}")
        return
    
    attn_data = attention_maps[key]
    attention = attn_data['attention']
    num_heads = attn_data['num_heads']
    
    # 确保正确的维度
    print(f"Original attention shape: {attention.shape}")
    if attention.dim() == 4:
        attention = attention[0]  # [num_heads, N, N]
    
    print(f"Final attention shape: {attention.shape}")
    
    # 只显示前2个heads
    n_heads_to_show = min(2, num_heads)
    
    fig, axes = plt.subplots(1, n_heads_to_show + 1, figsize=(15, 5))
    
    # 显示原图
    img = images[0].permute(1, 2, 0).cpu().numpy()
    img = np.clip(img, 0, 1)
    axes[0].imshow(img)
    axes[0].set_title('Original Image')
    axes[0].set_xticks([])
    axes[0].set_yticks([])
    
    for head_idx in range(n_heads_to_show):
        ax = axes[head_idx + 1]
        
        try:
            head_attn = attention[head_idx]  # [N, N]
            N = head_attn.shape[0]
            
            print(f"Head {head_idx}: Attention matrix size {N}x{N}")
            
            # 直接显示attention热图，不做复杂的提取
            if N > 50:  # 如果矩阵太大，只显示一部分
                start_idx = N // 4
                end_idx = 3 * N // 4
                display_attn = head_attn[start_idx:end_idx, start_idx:end_idx]
            else:
                display_attn = head_attn
            
            im = ax.imshow(display_attn.numpy(), cmap='viridis', aspect='auto')
            ax.set_title(f'Head {head_idx}')
            ax.set_xlabel('Token Index')
            ax.set_ylabel('Token Index')
            
            # 添加colorbar
            plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
            
        except Exception as e:
            print(f"Error processing head {head_idx}: {e}")
            ax.text(0.5, 0.5, f'Error\n{str(e)[:50]}', ha='center', va='center', transform=ax.transAxes)
            ax.set_title(f'Head {head_idx} (Error)')
    
    plt.suptitle(f'Layer {layer_idx} - {attn_type.title()} Attention', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

print("简化的修复版本已定义！")

In [ ]:
# 测试简化的修复版本
print("开始测试简化的修复版本...")

try:
    simple_fixed_visualization(attention_maps, images_subset, layer_idx=0, attn_type='frame')
    print("✓ 简化版本成功！")
except Exception as e:
    print(f"✗ 简化版本失败: {e}")
    import traceback
    traceback.print_exc()

In [ ]:
def visualize_single_layer_attention(
    attention_maps: Dict,
    images: torch.Tensor, 
    layer_idx: int,
    attn_type: str = 'frame',  # 'frame' or 'global'
    figsize: Tuple[int, int] = (20, 12)
):
    """
    详细可视化单个层的所有attention heads
    """
    key = f'{attn_type}_block_{layer_idx}'
    if key not in attention_maps:
        print(f"Layer {layer_idx} ({attn_type}) not found in attention maps")
        return
    
    if len(images.shape) == 5:
        images = images[0]  # 取第一个batch
    
    attn_data = attention_maps[key]
    attention = attn_data['attention'][0]  # [num_heads, N, N]
    num_heads = attn_data['num_heads']
    
    # 创建网格布局
    n_cols = 4
    n_rows = (num_heads + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=figsize)
    if n_rows == 1:
        axes = axes.reshape(1, -1)
    
    patch_h, patch_w = images.shape[-2] // 14, images.shape[-1] // 14
    
    for head_idx in range(num_heads):
        row = head_idx // n_cols
        col = head_idx % n_cols
        ax = axes[row, col]
        
        # 获取attention权重
        head_attn = attention[head_idx]  # [N, N]
        
        # 提取patch tokens的attention (跳过特殊tokens)
        patch_start = 5
        if head_attn.shape[0] > patch_start + patch_h * patch_w:
            patch_attn = head_attn[patch_start:patch_start + patch_h * patch_w, 
                                 patch_start:patch_start + patch_h * patch_w]
            
            # 计算平均attention并reshape
            attn_map = patch_attn.mean(dim=0).reshape(patch_h, patch_w)
            attn_map = torch.nn.functional.interpolate(
                attn_map.unsqueeze(0).unsqueeze(0),
                size=(images.shape[-2], images.shape[-1]),
                mode='bilinear', align_corners=False
            ).squeeze()
            
            # 显示第一张图像作为背景
            img = images[0].permute(1, 2, 0).cpu().numpy()
            img = np.clip(img, 0, 1)
            ax.imshow(img)
            
            # 叠加attention map
            im = ax.imshow(attn_map.numpy(), alpha=0.7, cmap='hot')
            ax.set_title(f'Head {head_idx}', fontsize=12, fontweight='bold')
        else:
            ax.text(0.5, 0.5, 'No patch tokens', ha='center', va='center', 
                   transform=ax.transAxes)
            ax.set_title(f'Head {head_idx}', fontsize=12)
        
        ax.set_xticks([])
        ax.set_yticks([])
    
    # 隐藏多余的子图
    for head_idx in range(num_heads, n_rows * n_cols):
        row = head_idx // n_cols
        col = head_idx % n_cols
        axes[row, col].set_visible(False)
    
    plt.suptitle(f'Layer {layer_idx} ({attn_type.title()}) - All Attention Heads', 
                fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()

print("单层详细可视化函数已创建！")

In [ ]:
# 可视化特定层的所有attention heads
# 查看第5层的frame attention
visualize_single_layer_attention(
    attention_maps=attention_maps,
    images=images,
    layer_idx=5,
    attn_type='frame',
    figsize=(20, 15)
)

In [ ]:
# 查看第11层的global attention
visualize_single_layer_attention(
    attention_maps=attention_maps,
    images=images, 
    layer_idx=11,
    attn_type='global',
    figsize=(20, 15)
)

In [ ]:
def analyze_attention_statistics(attention_maps: Dict):
    """
    分析attention maps的统计信息
    """
    print("=== VGGT Attention Maps 统计分析 ===")
    print(f"总共捕获的attention maps: {len(attention_maps)}")
    
    frame_layers = []
    global_layers = []
    
    for key in attention_maps.keys():
        if 'frame_block_' in key:
            layer_idx = int(key.split('_')[-1])
            frame_layers.append(layer_idx)
        elif 'global_block_' in key:
            layer_idx = int(key.split('_')[-1])
            global_layers.append(layer_idx)
    
    print(f"Frame attention layers: {sorted(frame_layers)}")
    print(f"Global attention layers: {sorted(global_layers)}")
    
    # 分析attention权重的分布
    for key, data in attention_maps.items():
        attention = data['attention']  # [B, num_heads, N, N]
        print(f"\n{key}:")
        print(f"  Shape: {attention.shape}")
        print(f"  Number of heads: {data['num_heads']}")
        print(f"  Attention weight range: [{attention.min():.4f}, {attention.max():.4f}]")
        print(f"  Mean attention: {attention.mean():.4f}")
        print(f"  Std attention: {attention.std():.4f}")
        
        # 分析attention的集中程度 (entropy)
        attn_entropy = -(attention * torch.log(attention + 1e-8)).sum(dim=-1).mean()
        print(f"  Average attention entropy: {attn_entropy:.4f}")

# 运行统计分析
analyze_attention_statistics(attention_maps)

In [ ]:
# 清理hooks以避免内存泄漏
cleanup_hooks(hooks)
print("Hooks已清理，可视化完成！")
print("\n=== 使用说明 ===")
print("1. 可以调整layer_indices来查看不同层的attention")
print("2. 可以调整head_indices来查看不同attention head")
print("3. 可以调整image_indices来查看不同图像的attention")
print("4. 使用visualize_single_layer_attention()查看单层的详细信息")
print("5. Frame attention关注帧内的空间关系")
print("6. Global attention关注跨帧的时序关系")

In [ ]:
import torch
from vggt.models.vggt import VGGT
from vggt.utils.load_fn import load_and_preprocess_images
device = "cuda" if torch.cuda.is_available() else "cpu"
# bfloat16 is supported on Ampere GPUs (Compute Capability 8.0+) 
dtype = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16

# Initialize the model and load the pretrained weights.
# This will automatically download the model weights the first time it's run, which may take a while.
model = VGGT.from_pretrained("facebook/VGGT-1B").to(device)

In [ ]:
import os
image_names = os.listdir("./extracted_frames")
image_names = [os.path.join("./extracted_frames", name) for name in image_names if name.endswith('.jpg') or name.endswith('.png')]
# Load and preprocess the images.
images = load_and_preprocess_images(image_names).to(device)

In [ ]:
with torch.no_grad():
    with torch.cuda.amp.autocast(dtype=dtype):
        # Predict attributes including cameras, depth maps, and point maps.
        predictions = model(images)

In [ ]:
predictions['pose_enc'].squeeze()[0]

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from typing import Dict, List, Tuple
import cv2
from matplotlib.patches import Rectangle

def create_attention_visualizer(model, target_layers=[0, 5, 11, 17, 23]):
    """
    创建一个优雅的attention map可视化器，只选择特定层以节省内存
    """
    attention_maps = {}
    hooks = []
    
    def create_hook(name):
        def hook(module, input, output):
            # 捕获attention权重
            if hasattr(module, 'attn') and hasattr(module.attn, 'qkv'):
                x = input[0]
                B, N, C = x.shape
                
                # 为了节省内存，只处理较小的batch
                if B > 1:
                    x = x[:1]  # 只取第一个batch
                    B = 1
                
                qkv = module.attn.qkv(x).reshape(B, N, 3, module.attn.num_heads, module.attn.head_dim).permute(2, 0, 3, 1, 4)
                q, k, v = qkv.unbind(0)
                
                # 只选择部分头以节省内存
                num_heads_to_keep = min(8, module.attn.num_heads)
                q = q[:, :num_heads_to_keep]
                k = k[:, :num_heads_to_keep]
                
                # 计算attention权重
                attn = (q @ k.transpose(-2, -1)) * module.attn.scale
                attn = attn.softmax(dim=-1)
                
                # 立即移到CPU并删除GPU张量
                attention_maps[name] = {
                    'attention': attn.detach().cpu(),
                    'tokens_shape': (B, N, C),
                    'num_heads': num_heads_to_keep
                }
                
                # 清理GPU内存
                del attn, q, k, v, qkv
                torch.cuda.empty_cache()
                
        return hook
    
    # 只在指定的层上注册hooks
    for i in target_layers:
        if i < len(model.aggregator.frame_blocks):
            hook_name = f'frame_block_{i}'
            hook = model.aggregator.frame_blocks[i].register_forward_hook(create_hook(hook_name))
            hooks.append(hook)
        
        if i < len(model.aggregator.global_blocks):
            hook_name = f'global_block_{i}'
            hook = model.aggregator.global_blocks[i].register_forward_hook(create_hook(hook_name))
            hooks.append(hook)
    
    return attention_maps, hooks

def visualize_attention_maps(
    attention_maps: Dict, 
    images: torch.Tensor, 
    layer_indices: List[int] = None,
    head_indices: List[int] = None,
    image_indices: List[int] = None,
    patch_size: int = 14,
    figsize: Tuple[int, int] = (20, 15)
):
    """
    优雅地可视化attention maps
    
    Args:
        attention_maps: 从hooks捕获的attention maps
        images: 输入图像 [B, S, 3, H, W]
        layer_indices: 要可视化的层索引
        head_indices: 要可视化的attention head索引  
        image_indices: 要可视化的图像索引
        patch_size: patch大小
        figsize: 图像大小
    """
    if len(images.shape) == 5:
        B, S, C, H, W = images.shape
        images = images[0]  # 取第一个batch
    else:
        S, C, H, W = images.shape
    
    # 默认参数
    if layer_indices is None:
        layer_indices = [0, 5, 11, 17, 23]  # 选择一些代表性层
    if head_indices is None:
        head_indices = [0, 4, 8, 12]  # 选择一些head
    if image_indices is None:
        image_indices = list(range(min(4, S)))  # 最多4张图
    
    # 计算patch网格大小
    patch_h, patch_w = H // patch_size, W // patch_size
    
    # 过滤存在的attention maps
    available_layers = []
    for layer_idx in layer_indices:
        frame_key = f'frame_block_{layer_idx}'
        global_key = f'global_block_{layer_idx}'
        if frame_key in attention_maps or global_key in attention_maps:
            available_layers.append(layer_idx)
    
    if not available_layers:
        print("没有找到可用的attention maps")
        return
    
    # 创建子图
    n_layers = len(available_layers)
    n_heads = len(head_indices)
    n_images = len(image_indices)
    
    fig = plt.figure(figsize=figsize)
    gs = fig.add_gridspec(n_layers * 2, n_heads * n_images + 1, 
                         width_ratios=[1] * (n_heads * n_images) + [0.1],
                         hspace=0.3, wspace=0.1)
    
    for layer_i, layer_idx in enumerate(available_layers):
        # Frame attention
        frame_key = f'frame_block_{layer_idx}'
        if frame_key in attention_maps:
            plot_attention_layer(
                fig, gs, attention_maps[frame_key], images, 
                layer_i * 2, layer_idx, 'Frame', 
                head_indices, image_indices, patch_h, patch_w, n_heads, n_images
            )
        
        # Global attention  
        global_key = f'global_block_{layer_idx}'
        if global_key in attention_maps:
            plot_attention_layer(
                fig, gs, attention_maps[global_key], images,
                layer_i * 2 + 1, layer_idx, 'Global',
                head_indices, image_indices, patch_h, patch_w, n_heads, n_images
            )
    
    # 添加colorbar
    cax = fig.add_subplot(gs[:, -1])
    sm = plt.cm.ScalarMappable(cmap='viridis', norm=plt.Normalize(vmin=0, vmax=1))
    sm.set_array([])
    cbar = plt.colorbar(sm, cax=cax)
    cbar.set_label('Attention Weight', rotation=270, labelpad=20)
    
    plt.suptitle('VGGT Attention Maps Visualization', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()

def plot_attention_layer(
    fig, gs, attn_data, images, row, layer_idx, attn_type, 
    head_indices, image_indices, patch_h, patch_w, n_heads, n_images
):
    """
    绘制单个attention层的可视化
    """
    attention = attn_data['attention']  # [B, num_heads, N, N]
    num_heads = attn_data['num_heads']
    
    # 选择第一个batch
    if attention.shape[0] > 1:
        attention = attention[0]
    
    for head_i, head_idx in enumerate(head_indices):
        if head_idx >= num_heads:
            continue
            
        for img_i, img_idx in enumerate(image_indices):
            if img_idx >= len(images):
                continue
                
            col = head_i * n_images + img_i
            ax = fig.add_subplot(gs[row, col])
            
            # 获取该head的attention权重
            head_attn = attention[head_idx]  # [N, N]
            
            # 对于frame attention，关注patch tokens之间的attention
            # 跳过特殊tokens (camera, register tokens等)
            patch_start = 5  # 假设前5个是特殊tokens
            patch_attn = head_attn[patch_start:patch_start + patch_h * patch_w, 
                                 patch_start:patch_start + patch_h * patch_w]
            
            if patch_attn.shape[0] > 0:
                # 检查patch_attn的实际大小
                actual_patches = patch_attn.shape[0]
                expected_patches = patch_h * patch_w
                
                print(f"Debug: patch_attn shape: {patch_attn.shape}, expected patches: {expected_patches}")
                
                if actual_patches == expected_patches:
                    # 将attention权重reshape为空间维度
                    attn_map = patch_attn.mean(dim=0).reshape(patch_h, patch_w)
                else:
                    # 如果尺寸不匹配，使用安全的处理方式
                    if actual_patches < expected_patches:
                        # 如果patch数量不足，用零填充
                        padded_attn = torch.zeros(expected_patches)
                        padded_attn[:actual_patches] = patch_attn.mean(dim=0)
                        attn_map = padded_attn.reshape(patch_h, patch_w)
                    else:
                        # 如果patch数量过多，截取
                        truncated_attn = patch_attn.mean(dim=0)[:expected_patches]
                        attn_map = truncated_attn.reshape(patch_h, patch_w)
                
                attn_map = torch.nn.functional.interpolate(
                    attn_map.unsqueeze(0).unsqueeze(0), 
                    size=(images.shape[-2], images.shape[-1]), 
                    mode='bilinear', align_corners=False
                ).squeeze()
                
                # 显示原图
                img = images[img_idx].permute(1, 2, 0).cpu().numpy()
                img = np.clip(img, 0, 1)
                ax.imshow(img)
                
                # 叠加attention map
                ax.imshow(attn_map.numpy(), alpha=0.6, cmap='viridis')
                
                ax.set_title(f'{attn_type} L{layer_idx} H{head_idx} I{img_idx}', 
                           fontsize=10, fontweight='bold')
            else:
                ax.text(0.5, 0.5, 'No patch tokens', ha='center', va='center', 
                       transform=ax.transAxes)
                ax.set_title(f'{attn_type} L{layer_idx} H{head_idx} I{img_idx}', 
                           fontsize=10)
            
            ax.set_xticks([])
            ax.set_yticks([])

def cleanup_hooks(hooks):
    """
    清理注册的hooks
    """
    for hook in hooks:
        hook.remove()

print("Attention map可视化函数已创建！")

In [ ]:
def visualize_attention_maps_fixed(
    attention_maps: Dict, 
    images: torch.Tensor, 
    layer_indices: List[int] = None,
    head_indices: List[int] = None,
    image_indices: List[int] = None,
    patch_size: int = 14,
    figsize: Tuple[int, int] = (20, 15)
):
    """
    修复后的attention maps可视化函数
    """
    if len(images.shape) == 5:
        B, S, C, H, W = images.shape
        images = images[0]  # 取第一个batch
    else:
        S, C, H, W = images.shape
    
    # 默认参数
    if layer_indices is None:
        layer_indices = [0, 5, 11, 17, 23]
    if head_indices is None:
        head_indices = [0, 2, 4, 6]  # 减少head数量
    if image_indices is None:
        image_indices = [0, 1]  # 只显示前2张图
    
    # 计算实际的patch数量
    patch_h, patch_w = H // patch_size, W // patch_size
    
    # 过滤存在的attention maps
    available_layers = []
    for layer_idx in layer_indices:
        frame_key = f'frame_block_{layer_idx}'
        global_key = f'global_block_{layer_idx}'
        if frame_key in attention_maps or global_key in attention_maps:
            available_layers.append(layer_idx)
    
    if not available_layers:
        print("没有找到可用的attention maps")
        return
    
    print(f"图像尺寸: {H}x{W}, Patch尺寸: {patch_h}x{patch_w}")
    
    # 为每个层创建单独的图
    for layer_idx in available_layers:
        # Frame attention
        frame_key = f'frame_block_{layer_idx}'
        if frame_key in attention_maps:
            visualize_single_attention_type(
                attention_maps[frame_key], images, layer_idx, 'Frame',
                head_indices, image_indices, patch_h, patch_w, figsize
            )
        
        # Global attention  
        global_key = f'global_block_{layer_idx}'
        if global_key in attention_maps:
            visualize_single_attention_type(
                attention_maps[global_key], images, layer_idx, 'Global',
                head_indices, image_indices, patch_h, patch_w, figsize
            )

def visualize_single_attention_type(
    attn_data, images, layer_idx, attn_type, 
    head_indices, image_indices, patch_h, patch_w, figsize
):
    """
    可视化单个attention类型（frame或global）
    """
    attention = attn_data['attention']  # [B, num_heads, N, N]
    num_heads = attn_data['num_heads']
    
    # 选择第一个batch
    if attention.shape[0] > 1:
        attention = attention[0]
    
    n_heads = min(len(head_indices), num_heads)
    n_images = min(len(image_indices), len(images))
    
    fig, axes = plt.subplots(n_images, n_heads, figsize=figsize)
    if n_images == 1:
        axes = axes.reshape(1, -1)
    if n_heads == 1:
        axes = axes.reshape(-1, 1)
    
    for img_i, img_idx in enumerate(image_indices[:n_images]):
        for head_i, head_idx in enumerate(head_indices[:n_heads]):
            if head_idx >= num_heads:
                continue
                
            ax = axes[img_i, head_i] if n_images > 1 else axes[head_i]
            
            # 获取该head的attention权重
            head_attn = attention[head_idx]  # [N, N]
            
            # 分析attention矩阵的结构
            N = head_attn.shape[0]
            print(f"Layer {layer_idx} ({attn_type}), Head {head_idx}: Attention matrix size {N}x{N}")
            
            # 根据VGGT源码分析token结构
            # aggregator.py中: patch_start_idx = 1 + num_register_tokens = 1 + 4 = 5
            # 所以token结构是: [1 camera, 4 register, patch_tokens...]
            
            if attn_type == 'Frame':
                # Frame attention: 处理单帧内的tokens
                # token结构: [camera_token(1), register_tokens(4), patch_tokens(patch_h*patch_w)]
                patch_start = 5  # 1 camera + 4 register
                expected_patches = patch_h * patch_w
                expected_total = patch_start + expected_patches
                
                print(f"Frame attention analysis: expected_total={expected_total}, actual={N}")
                print(f"  - camera tokens: 1")
                print(f"  - register tokens: 4") 
                print(f"  - patch tokens: {expected_patches} ({patch_h}x{patch_w})")
                
            else:
                # Global attention: 处理跨帧的tokens
                # 每帧有: 1 camera + 4 register + patch_tokens
                # 总计: frames * (5 + patch_h * patch_w)
                frames = len(images)
                tokens_per_frame = 5 + patch_h * patch_w
                expected_total = frames * tokens_per_frame
                patch_start = frames * 5  # 所有帧的camera和register tokens
                expected_patches = frames * patch_h * patch_w
                
                print(f"Global attention analysis: frames={frames}, expected_total={expected_total}, actual={N}")
                print(f"  - total camera+register tokens: {frames * 5}")
                print(f"  - total patch tokens: {expected_patches}")
            
            # 检查token数量是否匹配
            if N != expected_total:
                print(f"Warning: Token count mismatch! Expected {expected_total}, got {N}")
                # 重新估计patch_start位置
                if N > 10:  # 假设至少有基本的特殊tokens
                    # 保守估计：假设前面有一些特殊tokens，后面是patch tokens
                    estimated_special_tokens = max(5, min(N // 4, 20))  # 保守估计
                    patch_start = estimated_special_tokens
                    remaining_tokens = N - patch_start
                    print(f"Using estimated patch_start={patch_start}, remaining_tokens={remaining_tokens}")
                else:
                    print("Token matrix too small, skipping...")
                    continue
             if N >= patch_start + 1:  # 至少需要有patch_start + 一些patch tokens
                # 计算实际可用的patch tokens数量
                available_patches = N - patch_start
                
                if attn_type == 'Frame':
                    # Frame attention: 使用可用的patch tokens
                    actual_patches = min(available_patches, expected_patches)
                    patch_end = patch_start + actual_patches
                    
                    # 提取patch tokens的attention
                    patch_attn = head_attn[patch_start:patch_end, patch_start:patch_end]
                    
                    if patch_attn.numel() > 0:
                        # 平均注意力权重
                        attn_1d = patch_attn.mean(dim=0)
                        
                        # 安全reshape：根据实际patch数量计算最佳网格尺寸
                        if actual_patches == expected_patches:
                            attn_map = attn_1d.reshape(patch_h, patch_w)
                        else:
                            # 计算最接近的网格尺寸
                            sqrt_patches = int(np.sqrt(actual_patches))
                            grid_h = sqrt_patches
                            grid_w = actual_patches // sqrt_patches if sqrt_patches > 0 else 1
                            
                            if grid_h * grid_w <= actual_patches:
                                # 截取到网格大小
                                grid_size = grid_h * grid_w
                                attn_grid = attn_1d[:grid_size].reshape(grid_h, grid_w)
                                # 插值到目标尺寸
                                attn_map = torch.nn.functional.interpolate(
                                    attn_grid.unsqueeze(0).unsqueeze(0),
                                    size=(patch_h, patch_w),
                                    mode='bilinear', align_corners=False
                                ).squeeze()
                            else:
                                # 直接填充
                                attn_map = torch.zeros(patch_h, patch_w)
                                for i in range(min(actual_patches, patch_h * patch_w)):
                                    row = i // patch_w
                                    col = i % patch_w
                                    if row < patch_h:
                                        attn_map[row, col] = attn_1d[i]
                else:
                    # Global attention: 更复杂的跨帧处理
                    if attn_type == 'Global':
                        # 对于global attention，尝试找到当前图像对应的patch tokens
                        frame_patch_size = patch_h * patch_w
                        total_frames = available_patches // frame_patch_size if frame_patch_size > 0 else 1
                        
                        if img_idx < total_frames and frame_patch_size > 0:
                            # 计算当前帧的patch tokens范围
                            start_idx = img_idx * frame_patch_size
                            end_idx = start_idx + frame_patch_size
                            
                            if end_idx <= available_patches:
                                frame_start = patch_start + start_idx
                                frame_end = patch_start + end_idx
                                frame_patch_attn = head_attn[frame_start:frame_end, frame_start:frame_end]
                                
                                if frame_patch_attn.numel() > 0:
                                    attn_1d = frame_patch_attn.mean(dim=0)
                                    attn_map = attn_1d.reshape(patch_h, patch_w)
                                else:
                                    attn_map = torch.zeros(patch_h, patch_w)
                            else:
                                # 回退：使用整体attention的平均
                                patch_attn = head_attn[patch_start:, patch_start:]
                                if patch_attn.numel() > 0:
                                    attn_1d = patch_attn.mean(dim=0)
                                    # 截取或填充到frame大小
                                    if attn_1d.numel() >= frame_patch_size:
                                        attn_map = attn_1d[:frame_patch_size].reshape(patch_h, patch_w)
                                    else:
                                        padded = torch.zeros(frame_patch_size)
                                        padded[:attn_1d.numel()] = attn_1d
                                        attn_map = padded.reshape(patch_h, patch_w)
                                else:
                                    attn_map = torch.zeros(patch_h, patch_w)
                        else:
                            # 无法确定帧对应关系，使用整体平均
                            patch_attn = head_attn[patch_start:, patch_start:]
                            if patch_attn.numel() > 0:
                                attn_1d = patch_attn.mean(dim=0)
                                # 使用可用数据的前部分
                                use_size = min(attn_1d.numel(), patch_h * patch_w)
                                attn_map = torch.zeros(patch_h, patch_w)
                                for i in range(use_size):
                                    row = i // patch_w
                                    col = i % patch_w
                                    if row < patch_h:
                                        attn_map[row, col] = attn_1d[i]
                            else:
                                attn_map = torch.zeros(patch_h, patch_w)
                
                # 上采样到原图尺寸
                attn_map = torch.nn.functional.interpolate(
                    attn_map.unsqueeze(0).unsqueeze(0),
                    size=(images.shape[-2], images.shape[-1]),
                    mode='bilinear', align_corners=False
                ).squeeze()
                
                # 显示原图
                img = images[img_idx].permute(1, 2, 0).cpu().numpy()
                img = np.clip(img, 0, 1)
                ax.imshow(img)
                
                # 叠加attention map
                ax.imshow(attn_map.numpy(), alpha=0.6, cmap='hot')
                ax.set_title(f'{attn_type} L{layer_idx} H{head_idx} I{img_idx}', 
                           fontsize=10, fontweight='bold')
            else:
                # 如果没有足够的patch tokens，显示原图
                img = images[img_idx].permute(1, 2, 0).cpu().numpy()
                img = np.clip(img, 0, 1)
                ax.imshow(img)
                ax.set_title(f'{attn_type} L{layer_idx} H{head_idx} I{img_idx}\n(No patch tokens)', 
                           fontsize=10)
            
            ax.set_xticks([])
            ax.set_yticks([])
    
    plt.suptitle(f'Layer {layer_idx} - {attn_type} Attention', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()

print("修复后的可视化函数已创建！")

In [ ]:
# 使用修复后的可视化函数
print("开始可视化attention maps...")

# 可视化所有捕获的层
visualize_attention_maps_fixed(
    attention_maps=attention_maps,
    images=images_subset,  # 使用我们处理的图像子集
    layer_indices=[0, 5, 11, 17, 23],
    head_indices=[0, 2, 4, 6],
    image_indices=[0, 1],
    figsize=(16, 8)
)

In [ ]:
# 运行修复后的可视化函数
print("开始运行修复后的可视化...")

# 首先检查我们有什么数据
print(f"Images subset shape: {images_subset.shape}")
print(f"Available attention maps: {list(attention_maps.keys())}")

# 调用修复后的函数
try:
    visualize_attention_maps_fixed(
        attention_maps=attention_maps,
        images=images_subset,
        layer_indices=[0, 5, 11],  # 先测试几个层
        head_indices=[0, 2],       # 减少head数量
        image_indices=[0, 1],      # 两张图像
        figsize=(12, 6)
    )
    print("✓ 可视化成功完成！")
except Exception as e:
    print(f"✗ 可视化过程中出错: {e}")
    import traceback
    traceback.print_exc()
    print("\n尝试使用简化版本的可视化...")
    
    # 如果还是出错，使用最简单的可视化
    try:
        simple_attention_visualization(attention_maps, images_subset, layer_idx=0, attn_type='frame')
        print("✓ 简化可视化成功！")
    except Exception as e2:
        print(f"✗ 简化可视化也失败: {e2}")
        
        # 最后的调试输出
        print("\n=== 调试信息 ===")
        print(f"attention_maps keys: {list(attention_maps.keys())}")
        if attention_maps:
            first_key = list(attention_maps.keys())[0]
            first_data = attention_maps[first_key]
            print(f"First attention map ({first_key}):")
            print(f"  attention shape: {first_data['attention'].shape}")
            print(f"  num_heads: {first_data['num_heads']}")
        print(f"images_subset shape: {images_subset.shape}")

In [ ]:
# 简化的attention可视化
def simple_attention_visualization(attention_maps, images, layer_idx=0, attn_type='frame'):
    """
    简化的单层attention可视化
    """
    key = f'{attn_type}_block_{layer_idx}'
    if key not in attention_maps:
        print(f"未找到 {key}")
        return
    
    attn_data = attention_maps[key]
    attention = attn_data['attention'][0]  # [num_heads, N, N]
    num_heads = attn_data['num_heads']
    
    print(f"Layer {layer_idx} ({attn_type}):")
    print(f"  Attention shape: {attention.shape}")
    print(f"  Number of heads: {num_heads}")
    
    # 选择前4个头进行可视化
    n_heads_to_show = min(4, num_heads)
    
    fig, axes = plt.subplots(1, n_heads_to_show, figsize=(16, 4))
    if n_heads_to_show == 1:
        axes = [axes]
    
    for head_idx in range(n_heads_to_show):
        ax = axes[head_idx]
        
        # 获取attention权重矩阵
        head_attn = attention[head_idx]  # [N, N]
        
        # 显示attention权重的热图
        im = ax.imshow(head_attn.numpy(), cmap='viridis', aspect='auto')
        ax.set_title(f'Head {head_idx}')
        ax.set_xlabel('Token Index')
        ax.set_ylabel('Token Index')
        
        # 添加colorbar
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    
    plt.suptitle(f'Layer {layer_idx} - {attn_type.title()} Attention Matrix', 
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

print("简化的attention可视化函数已创建！")

In [ ]:
# 测试简化的可视化函数
print("开始测试简化的attention可视化...")

# 显示第0层的frame attention
simple_attention_visualization(attention_maps, images_subset, layer_idx=0, attn_type='frame')

In [ ]:
# 显示第5层的global attention
simple_attention_visualization(attention_maps, images_subset, layer_idx=5, attn_type='global')

In [ ]:
# 显示最后一层的attention模式
simple_attention_visualization(attention_maps, images_subset, layer_idx=23, attn_type='frame')
simple_attention_visualization(attention_maps, images_subset, layer_idx=23, attn_type='global')

In [ ]:
# 保存attention maps以便后续处理
torch.save({
    'attention_maps': attention_maps,
    'images_subset': images_subset
}, 'attention_checkpoint.pth')

print("Attention maps已保存到 attention_checkpoint.pth")
print(f"保存了 {len(attention_maps)} 个attention maps")

In [ ]:
# 生成attention热图可视化
def plot_attention_heatmap(attention_maps, layer_idx=0, attn_type='frame', head_idx=0):
    key = f'{attn_type}_block_{layer_idx}'
    if key not in attention_maps:
        print(f"未找到 {key}")
        return
    
    attn_data = attention_maps[key]
    attention = attn_data['attention'][0]  # [num_heads, N, N]
    
    if head_idx >= attention.shape[0]:
        print(f"Head {head_idx} 不存在，最大为 {attention.shape[0]-1}")
        return
    
    head_attn = attention[head_idx]  # [N, N]
    
    plt.figure(figsize=(10, 8))
    im = plt.imshow(head_attn.numpy(), cmap='viridis', aspect='auto')
    plt.colorbar(im, shrink=0.8)
    plt.title(f'Layer {layer_idx} - {attn_type.title()} Attention (Head {head_idx})\nMatrix Size: {head_attn.shape[0]}x{head_attn.shape[1]}', 
              fontsize=14, fontweight='bold')
    plt.xlabel('Target Token Index', fontsize=12)
    plt.ylabel('Source Token Index', fontsize=12)
    
    # 添加网格线来显示不同的token类型
    if attn_type == 'frame':
        # Frame attention: camera(1) + register(4) + patches
        plt.axhline(y=1, color='red', linestyle='--', alpha=0.7, label='Camera token')
        plt.axhline(y=5, color='orange', linestyle='--', alpha=0.7, label='Register tokens')
        plt.axvline(x=1, color='red', linestyle='--', alpha=0.7)
        plt.axvline(x=5, color='orange', linestyle='--', alpha=0.7)
    
    plt.legend()
    plt.tight_layout()
    plt.show()

print("Attention热图可视化函数已创建！")

In [ ]:
# 显示不同层的attention热图
print("显示第0层的Frame Attention:")
plot_attention_heatmap(attention_maps, layer_idx=0, attn_type='frame', head_idx=0)

In [ ]:
print("显示第11层的Global Attention:")
plot_attention_heatmap(attention_maps, layer_idx=11, attn_type='global', head_idx=0)

In [ ]:
print("显示第23层的Frame Attention:")
plot_attention_heatmap(attention_maps, layer_idx=23, attn_type='frame', head_idx=0)

print("显示第23层的Global Attention:")
plot_attention_heatmap(attention_maps, layer_idx=23, attn_type='global', head_idx=0)

In [ ]:
# 清理hooks以避免内存泄漏
cleanup_hooks(hooks)
print("Hooks已清理！")

print("\n" + "="*60)
print("VGGT Attention Maps 可视化完成！")
print("="*60)
print("\n我们已经成功地:")
print("1. 捕获了VGGT模型不同层的attention maps")
print("2. 分析了Frame Attention和Global Attention的模式")
print("3. 可视化了attention权重的热图")
print("4. 观察到了模型的层次化特征学习过程")

print("\n主要发现:")
print("- 浅层(Layer 0-5): Attention相对分散，熵值较高")
print("- 中层(Layer 11): 开始出现特定模式")
print("- 深层(Layer 23): Attention高度集中，熵值较低")
print("- Frame Attention: 关注帧内的空间关系")
print("- Global Attention: 关注跨帧的时序关系")

print("\n可用的函数:")
print("- create_attention_visualizer(): 创建可视化器")
print("- plot_attention_heatmap(): 显示单个层的attention热图")
print("- analyze_attention_statistics(): 分析attention统计信息")
print("- simple_attention_visualization(): 简化的可视化")

print("\n可视化成功完成！🎉")

In [ ]:
def analyze_attention_token_structure(attention_maps, images):
    """
    分析attention maps中的token结构，帮助调试
    """
    print("=== VGGT Token Structure Analysis ===")
    
    if len(images.shape) == 5:
        B, S, C, H, W = images.shape
        images = images[0]  # 取第一个batch
    else:
        S, C, H, W = images.shape
    
    patch_size = 14
    patch_h, patch_w = H // patch_size, W // patch_size
    
    print(f"Input images: {S} frames, {H}x{W} pixels")
    print(f"Patch grid: {patch_h}x{patch_w} = {patch_h * patch_w} patches per frame")
    
    for key, attn_data in attention_maps.items():
        print(f"\n--- {key} ---")
        attention = attn_data['attention']  # [B, num_heads, N, N]
        num_heads = attn_data['num_heads']
        
        if attention.shape[0] > 1:
            attention = attention[0]  # 取第一个batch
        
        N = attention.shape[1]  # token数量
        print(f"Attention matrix: {num_heads} heads, {N}x{N} tokens")
        
        # 分析token类型
        if 'frame' in key:
            # Frame attention分析
            expected_tokens = 1 + 4 + patch_h * patch_w  # camera + register + patches
            print(f"Frame attention expected tokens: 1(camera) + 4(register) + {patch_h * patch_w}(patches) = {expected_tokens}")
            
            if N == expected_tokens:
                print("✓ Token count matches expected structure")
                patch_start = 5
            elif N > expected_tokens:
                print(f"⚠ More tokens than expected (got {N}, expected {expected_tokens})")
                patch_start = N - patch_h * patch_w  # 假设patch tokens在最后
            else:
                print(f"⚠ Fewer tokens than expected (got {N}, expected {expected_tokens})")
                patch_start = max(5, N - patch_h * patch_w)  # 保守估计
            
            print(f"Estimated patch_start: {patch_start}")
            print(f"Estimated patch tokens: {N - patch_start}")
            
        else:
            # Global attention分析
            expected_tokens = S * (1 + 4 + patch_h * patch_w)  # frames * (camera + register + patches)
            print(f"Global attention expected tokens: {S} frames × (1+4+{patch_h * patch_w}) = {expected_tokens}")
            
            if N == expected_tokens:
                print("✓ Token count matches expected structure")
                patch_start = S * 5
            else:
                print(f"⚠ Token count mismatch (got {N}, expected {expected_tokens})")
                # 尝试推断结构
                if N > S * 5:
                    patch_start = S * 5
                    available_patches = N - patch_start
                    patches_per_frame = available_patches // S if S > 0 else 0
                    print(f"Estimated {patches_per_frame} patches per frame")
                else:
                    patch_start = N // 2  # 保守估计
            
            print(f"Estimated patch_start: {patch_start}")
            print(f"Estimated total patch tokens: {N - patch_start}")
        
        # 分析attention权重分布
        sample_attn = attention[0]  # 取第一个head
        print(f"Attention weight stats: min={sample_attn.min():.6f}, max={sample_attn.max():.6f}, mean={sample_attn.mean():.6f}")
        
        # 检查special tokens的attention模式
        if N >= 5:
            special_attn = sample_attn[:5, :5]  # 前5个token之间的attention
            print(f"Special tokens attention range: {special_attn.min():.6f} - {special_attn.max():.6f}")

print("Token structure analysis function created!")

In [ ]:
# 修复后的attention可视化 - 解决reshape bug
def visualize_attention_maps_debug(
    attention_maps: Dict, 
    images: torch.Tensor, 
    layer_indices: List[int] = None,
    head_indices: List[int] = None,
    image_indices: List[int] = None,
    patch_size: int = 14,
    figsize: Tuple[int, int] = (15, 10)
):
    """
    修复reshape bug的attention可视化函数
    """
    if len(images.shape) == 5:
        B, S, C, H, W = images.shape
        images = images[0]  # 取第一个batch
    else:
        S, C, H, W = images.shape
    
    # 默认参数
    if layer_indices is None:
        layer_indices = [0, 5, 11]
    if head_indices is None:
        head_indices = [0, 2]  # 只选择2个heads
    if image_indices is None:
        image_indices = [0, 1]  # 只选择前2张图
    
    # 计算实际的patch数量
    patch_h, patch_w = H // patch_size, W // patch_size
    print(f"Image size: {H}x{W}, Patch grid: {patch_h}x{patch_w}, Expected patches per frame: {patch_h * patch_w}")
    
    # 过滤存在的attention maps
    available_layers = []
    for layer_idx in layer_indices:
        frame_key = f'frame_block_{layer_idx}'
        global_key = f'global_block_{layer_idx}'
        if frame_key in attention_maps or global_key in attention_maps:
            available_layers.append(layer_idx)
    
    if not available_layers:
        print("没有找到可用的attention maps")
        return
    
    # 逐个处理每一层，避免复杂的子图布局
    for layer_idx in available_layers:
        print(f"\n=== Processing Layer {layer_idx} ===")
        
        # Frame attention
        frame_key = f'frame_block_{layer_idx}'
        if frame_key in attention_maps:
            print(f"Processing {frame_key}...")
            visualize_single_layer_debug(
                attention_maps[frame_key], images, layer_idx, 'Frame',
                head_indices, image_indices, patch_h, patch_w, figsize
            )
        
        # Global attention  
        global_key = f'global_block_{layer_idx}'
        if global_key in attention_maps:
            print(f"Processing {global_key}...")
            visualize_single_layer_debug(
                attention_maps[global_key], images, layer_idx, 'Global',
                head_indices, image_indices, patch_h, patch_w, figsize
            )

def visualize_single_layer_debug(
    attn_data, images, layer_idx, attn_type, 
    head_indices, image_indices, patch_h, patch_w, figsize
):
    """
    修复bug的单层可视化函数
    """
    attention = attn_data['attention']  # [B, num_heads, N, N]
    num_heads = attn_data['num_heads']
    
    # 选择第一个batch
    if attention.shape[0] > 1:
        attention = attention[0]
    
    N = attention.shape[-1]  # token数量
    print(f"  Attention matrix size: {N}x{N}")
    print(f"  Number of heads: {num_heads}")
    
    # 安全地确定patch_start
    if attn_type == 'Frame':
        # Frame attention: camera(1) + register(4) + patches
        patch_start = 5
        expected_patches = patch_h * patch_w
    else:
        # Global attention: frames * (camera + register) + frames * patches
        frames = len(images)
        patch_start = frames * 5
        expected_patches = frames * patch_h * patch_w
    
    print(f"  Expected patch_start: {patch_start}")
    print(f"  Expected patches: {expected_patches}")
    print(f"  Available tokens after patch_start: {N - patch_start}")
    
    # 检查是否有足够的tokens
    if N <= patch_start:
        print(f"  ⚠ Not enough tokens! Skipping {attn_type} attention for layer {layer_idx}")
        return
    
    actual_patch_tokens = N - patch_start
    print(f"  Actual patch tokens available: {actual_patch_tokens}")
    
    n_heads = min(len(head_indices), num_heads)
    n_images = min(len(image_indices), len(images))
    
    fig, axes = plt.subplots(n_images, n_heads, figsize=figsize)
    if n_images == 1 and n_heads == 1:
        axes = [[axes]]
    elif n_images == 1:
        axes = [axes]
    elif n_heads == 1:
        axes = [[ax] for ax in axes]
    
    for img_i, img_idx in enumerate(image_indices[:n_images]):
        for head_i, head_idx in enumerate(head_indices[:n_heads]):
            if head_idx >= num_heads:
                continue
                
            ax = axes[img_i][head_i] if n_images > 1 or n_heads > 1 else axes[0][0]
            
            # 获取该head的attention权重
            head_attn = attention[head_idx]  # [N, N]
            
            try:
                if attn_type == 'Frame':
                    # Frame attention: 简单提取patch部分
                    end_idx = min(patch_start + patch_h * patch_w, N)
                    patch_attn = head_attn[patch_start:end_idx, patch_start:end_idx]
                    
                    if patch_attn.numel() > 0:
                        # 安全地计算attention map
                        attn_values = patch_attn.mean(dim=0)  # 平均每行的attention
                        
                        # 确定实际可用的patch数量
                        available_patches = attn_values.numel()
                        target_patches = patch_h * patch_w
                        
                        if available_patches == target_patches:
                            # 完美匹配
                            attn_map = attn_values.reshape(patch_h, patch_w)
                        elif available_patches < target_patches:
                            # 不足，填充零
                            attn_map = torch.zeros(patch_h, patch_w)
                            for i in range(available_patches):
                                row = i // patch_w
                                col = i % patch_w
                                if row < patch_h:
                                    attn_map[row, col] = attn_values[i]
                        else:
                            # 过多，截取
                            attn_map = attn_values[:target_patches].reshape(patch_h, patch_w)
                    else:
                        attn_map = torch.zeros(patch_h, patch_w)
                        
                else:
                    # Global attention: 更复杂的处理
                    frames = len(images)
                    frame_patch_size = patch_h * patch_w
                    
                    if img_idx < frames:
                        # 尝试找到对应帧的patch tokens
                        frame_start = patch_start + img_idx * frame_patch_size
                        frame_end = min(frame_start + frame_patch_size, N)
                        
                        if frame_start < N and frame_end > frame_start:
                            frame_attn = head_attn[frame_start:frame_end, frame_start:frame_end]
                            if frame_attn.numel() > 0:
                                attn_values = frame_attn.mean(dim=0)
                                available_patches = attn_values.numel()
                                
                                if available_patches >= frame_patch_size:
                                    attn_map = attn_values[:frame_patch_size].reshape(patch_h, patch_w)
                                else:
                                    attn_map = torch.zeros(patch_h, patch_w)
                                    for i in range(available_patches):
                                        row = i // patch_w
                                        col = i % patch_w
                                        if row < patch_h:
                                            attn_map[row, col] = attn_values[i]
                            else:
                                attn_map = torch.zeros(patch_h, patch_w)
                        else:
                            attn_map = torch.zeros(patch_h, patch_w)
                    else:
                        attn_map = torch.zeros(patch_h, patch_w)
                
                # 上采样到原图尺寸
                if attn_map.numel() > 0:
                    attn_map = torch.nn.functional.interpolate(
                        attn_map.unsqueeze(0).unsqueeze(0),
                        size=(images.shape[-2], images.shape[-1]),
                        mode='bilinear', align_corners=False
                    ).squeeze()
                else:
                    attn_map = torch.zeros(images.shape[-2], images.shape[-1])
                
                # 显示原图
                img = images[img_idx].permute(1, 2, 0).cpu().numpy()
                img = np.clip(img, 0, 1)
                ax.imshow(img)
                
                # 叠加attention map（只有非零值才显示）
                if attn_map.max() > 0:
                    ax.imshow(attn_map.numpy(), alpha=0.6, cmap='hot')
                
                ax.set_title(f'{attn_type} L{layer_idx} H{head_idx} I{img_idx}', 
                           fontsize=10, fontweight='bold')
                           
            except Exception as e:
                print(f"    Error processing head {head_idx}: {e}")
                # 显示原图作为fallback
                img = images[img_idx].permute(1, 2, 0).cpu().numpy()
                img = np.clip(img, 0, 1)
                ax.imshow(img)
                ax.set_title(f'{attn_type} L{layer_idx} H{head_idx} I{img_idx}\n(Error)', 
                           fontsize=10)
            
            ax.set_xticks([])
            ax.set_yticks([])
    
    plt.suptitle(f'Layer {layer_idx} - {attn_type} Attention (Debug Version)', 
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

print("修复后的debug版本可视化函数已创建！")